# Markov switching com trend following 
- Primeiramente iremos escolher as acoes que iremos utilizar.
Utilizaremos as ações que tem melhor relacao entre drift e volatilidade.
Sendo importante para o nosso modelo, essa relação para poder identificar uma possivel tendencia
- Para evitar look ahead bias, iremos utilizar dados de (2010 - 2018)

In [2]:
import numpy as np
import pandas as pd
import yfinance as yf

# Grandes empresas da B3 no início de 2010 (tickers atuais no Yahoo).
# Ficam de fora por falta de dados (viés de sobrevivência, citar como limitação):
# OGXP3 (OGX), RDCD3 (Redecard), CRUZ3 (Souza Cruz), AMIL3 (Amil), OIBR4 (Oi)
empresas_brasileiras = [
    "PETR4", "VALE3", "ITUB4", "BBDC4", "BBAS3", "ABEV3", "ITSA4", "B3SA3",
    "GGBR4", "CSNA3", "USIM5", "VIVT3", "CMIG4", "SBSP3", "CPFE3",
    "PCAR3", "TIMS3", "WEGE3", "LREN3", "SUZB3", "RENT3",
]
tickers = [t + ".SA" for t in empresas_brasileiras]

# Só o período de treino
precos = yf.download(tickers, start="2010-01-01", end="2018-12-31",
                     auto_adjust=True, progress=False)["Close"]

# Remove ações sem histórico suficiente (exige 90% dos dias com dado)
cobertura = precos.notna().mean()
removidas = cobertura[cobertura < 0.9].index.tolist()
if removidas:
    print("Removidas por falta de dados:", removidas)
precos = precos.drop(columns=removidas)

# Drift e volatilidade anualizados (retornos log)
retornos = np.log(precos).diff()
drift = retornos.mean() * 252
vol = retornos.std() * np.sqrt(252)

ranking = pd.DataFrame({"drift": drift, "vol": vol, "mu/sigma": drift / vol})
ranking = ranking.sort_values("mu/sigma", ascending=False)

selecionados = ranking.head(3).index.tolist()

display(ranking)
print("Selecionados:", selecionados)

,drift,vol,mu/sigma
Ticker,,,
LREN3.SA,0.221326,0.325009,0.680984
ABEV3.SA,0.143958,0.228541,0.629902
RENT3.SA,0.201483,0.340855,0.591110
SUZB3.SA,0.108487,0.185125,0.586019
WEGE3.SA,0.155859,0.272944,0.571030
SBSP3.SA,0.154166,0.333481,0.462294
ITUB4.SA,0.116569,0.301764,0.386291
ITSA4.SA,0.114196,0.300913,0.379498
CPFE3.SA,0.093825,0.262790,0.357034


Selecionados: ['LREN3.SA', 'ABEV3.SA', 'RENT3.SA']


# Escolha das ações 
## Assim escolhemos como ações:

- MCD

- AMZN

- APPL

 

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import backtrader as bt
import yfinance as yf

RF = 4.0                       # taxa livre de risco dos EUA (% a.a.), no lugar do CDI
tickers = selecionados         # ações escolhidas na célula anterior
date_from = "2010-01-01"
date_to = "2026-07-31"

prices = {}
for ticker in tickers:
    prices[ticker] = yf.download(ticker, start=date_from, end=date_to, interval='1d', auto_adjust=True)

    # Remove o ticker do index
    prices[ticker] = prices[ticker].droplevel(1, axis=1)
    